# Exploratory Data Analysis

In [ ]:
# Imports & settings
from foodcast.imports import *
notebook_settings() 

# Load Data
location_ids_by_coverage = load_loc_ids()
locations, before_after_details_true, items_tagged, customers = load_static()
sales_and_menu_data = load_sales()

## Static Reference Data Exploration

### Menu Stats

In [ ]:
print(f'Total number of menu items: {items_tagged.shape[0]}')
print(f'Median number of menu items for a restaurant: {round(items_tagged.groupby("location_id")["item_name"].count().median())}')

### Menu Visuals

In [ ]:
# # How many categories to include?
# topn = 20

# # Irrelevant dish categories
# irrelevant = 2

# ## Colors for the bars
# colors1 = ["#ef8a62", "#67a9cf"]
# colors2 = ["#ff6961", "#aec6cf", "#77dd77"]

# # items_tagged needs is_alcohol

# # Calculate the counts for each combination
# pivot_df1 = items_tagged.groupby(['is_plant_based', 'is_alcohol'], observed=True).size().unstack(fill_value=0) # is_alcohol should be the second group by
# pivot_df1 = pivot_df1.loc[pivot_df1.sum(axis=1).sort_values(ascending=False).index] # Sort

# # Calculate the counts for each combination
# pivot_df2 = items_tagged.groupby(['item_type', 'is_plant_based'], observed=True).size().unstack(fill_value=0)
# pivot_df2 = pivot_df2.loc[pivot_df2.sum(axis=1).sort_values(ascending=False).index] # Sort

# # Calculate the counts for each combination
# pivot_df3 = items_tagged.groupby(['dish_category', 'is_plant_based'], observed=True).size().unstack(fill_value=0)
# pivot_df3 = pivot_df3.loc[pivot_df3.sum(axis=1).sort_values(ascending=False).index[:topn]] # Sort

# # Calculate the counts for each combination
# pivot_df4 = items_tagged.groupby(['dish_category', 'is_plant_based'], observed=True).size().unstack(fill_value=0)
# pivot_df4 = pivot_df4.loc[pivot_df4.sum(axis=1).sort_values(ascending=False).index[irrelevant:topn+irrelevant]] # Sort

# # Add to lists
# dfs = [pivot_df1, pivot_df2, pivot_df3, pivot_df4]
# color_groups = [colors1, colors2, colors2, colors2]
# titles = ["Plant Based", "Meal Type Counts", "Dish Category Counts", "Dish Category Counts w/o Alcohol or Unknowns"]
# xlabs = ["Plant Based", "Meal Type", "Dish Category", "Dish Category"]
# legends = ["Is Alcohol?", "Is Plant Based?", "Is Plant Based?", "Is Plant Based?"]
# coordinates = [(0,0),(0,1),(1,0),(1,1)]

# # Create a 2x2 grid of subplots
# fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# # Zip together for looping
# data_for_visual = zip(dfs, color_groups, titles, xlabs, legends, coordinates)
# for i, data_tuple in enumerate(data_for_visual):

#     # Unpack
#     df, colors, title, xlab, legend, coordinate = data_tuple
#     a, b = coordinate

#     # Initialize a series of zeros with the same index
#     left_start = pd.Series(0, index=df.index)

#     # Loop through every variable to be stacked
#     for j, col in enumerate(df.columns):

#         # Stack up multiple bar charts
#         sns.barplot(x=df[col], 
#                     y=df.index.tolist(), 
#                     left=left_start, 
#                     color=colors[j], 
#                     label=col,
#                     ax=axes[a,b])
        
#         # Start the next level exactly where the last one
#         left_start = left_start + df[col]

#     axes[a,b].set_ylabel('Count')
#     axes[a,b].set_xlabel(xlab)
#     #axes[a,b].set_yticks(range(df.index.size))
#     #axes[a,b].set_yticklabels(labels=df.index.to_series().str.capitalize().tolist())
#     axes[a,b].set_title(title)
#     axes[a,b].legend(title=legend)

# # Adjust layout
# plt.tight_layout()

# # Create a margin for a global title
# plt.subplots_adjust(top=0.88)

# # Add global title
# fig.suptitle('                   Visual Summary of Menu Items', fontsize=16)

# # Save plot
# plt.savefig('visuals/Menu Summary Stats.png', bbox_inches='tight')

# # Show the plots
# plt.show()

## Restaurant Sales Data Exploration

Totals

In [ ]:
# Recheck total number of entries
sales_items_total = 0
sales_data_total = 0
sales_transactions_total = 0
for loc_id, df in sales_and_menu_data.items():
    sales_items_total += df['item_quantity'].sum()
    sales_data_total += df.shape[0]
    sales_transactions_total += df['order_id'].nunique()

# Print
print(f'Total number of items sold: {sales_items_total:,}')
print(f'Total number of sales entries (row): {sales_data_total:,}')
print(f'Total number of transactions: {sales_transactions_total:,}')

Timeframes

In [ ]:
list_of_timeframes = []

# Find the time difference
for location_id, df in tqdm(sales_and_menu_data.items()):

    # Find the time difference
    timedelta = df.index[-1] - df.index[0]

    # Convert to days, then to years
    years = timedelta.days / 365.25

    # Append
    list_of_timeframes.append(years)

# Turn into an np array for finding median, mean, and std
timeframes = np.array(list_of_timeframes)

# Display
print("Median: {:.2f} year range".format(np.median(timeframes)))
print("Mean: {:.2f} year range".format(np.mean(timeframes)))
print("SD: {:.2f} years".format(np.std(timeframes)))
print("Restaurants with less than a 2 year range: {}".format((timeframes < 2).sum()))

In [ ]:
loc_id = location_ids_by_coverage[1]
exposure_date = pd.to_datetime(before_after_details_true.loc[loc_id, 'cross_over_date'])
plot_time_series(sales_and_menu_data[loc_id],
                 exposure=exposure_date,
                 freq='D',
                 truncate=False)
plot_time_series((sales_and_menu_data[loc_id]
                  .query('customer_id.isin(@before_after_customers)')
                  ),
                 exposure=exposure_date,
                 freq='D',
                 truncate=False)
plot_time_series_subset(sales_and_menu_data[loc_id],
                 exposure=exposure_date,
                 freq='W',
                 truncate=False,
                 normalize=False)
plot_time_series_subset((sales_and_menu_data[loc_id]
                  .query('customer_id.isin(@before_after_customers)')
                  ),
                 exposure=exposure_date,
                 freq='W',
                 truncate=True,
                 normalize=True)
plt.show()

### Promotional Items

In [ ]:
promo_match_list = []

for loc_id, df in sales_and_menu_data.items():

    # Looking for promo
    promo_item = before_after_details_true.loc[loc_id, 'first_plant_based_mention']
    cross_over_date = before_after_details_true.loc[loc_id, 'cross_over_date']
    promo_df = df.query('item_name == @promo_item')
    ever_found = not promo_df.empty


    # Actual first
    plant_based = df.query('is_plant_based == "Yes"')
    first_plant_based = plant_based['item_name'].iloc[0]
    its_date = plant_based.index[0]

    # Do they match?
    is_first = promo_item.lower() == first_plant_based.lower()

    row = {'location_id': loc_id, 'promo_item': promo_item, 'cross_over_date': cross_over_date, 'ever_found': ever_found, 'is_first': is_first, 'first_plant_based': first_plant_based, 'its_date': its_date}

    promo_match_list.append(row)
    
promo_match = pd.DataFrame(promo_match_list)

display(promo_match)

### Sales Visuals

In [ ]:
num_plots = 4#len(sales_and_menu_data)
cols = 4 
rows = num_plots

# Create a figure with multiple subplots
fig, axs = plt.subplots(rows, cols, figsize=(15, 5 * rows))
axs = axs.flatten()  # Flatten the array for easy indexing

# How many dishes to include
top_dishes_number = 20

# How many characters to include of each dish
num_characters = 22

# Lower the length of menu item names for the visual
items_tagged_copy = (items_tagged
        .assign(item_name=items_tagged.item_name.str.slice(0,num_characters).str.strip('.'))
        .drop_duplicates(subset=['location_id','item_name']))

for i, (location_id, data) in tqdm(enumerate(list(sales_and_menu_data.items())[0:4])):

    # Filter to the relevant restaurant
    items_tagged_res = items_tagged_copy.query('location_id == @location_id')

    # Just for the visual
    df = data.assign(item_name = data.item_name.str.slice(0,num_characters))
    
    # Colors for visual
    colors_df = pd.DataFrame(["#ff6961", "#aec6cf", "#77dd77"], columns=['colors'], index=['No','Unsure','Yes'])
    colors_df = pd.merge(items_tagged_res[['item_name', 'is_plant_based']], colors_df,
                  left_on='is_plant_based', right_index=True,
                  how='left')[['item_name','colors']]
    colors_df.set_index('item_name', drop=True, inplace=True)

    # Plant based
    plant_df = df[df['is_plant_based'] == 'Yes']

    # Get the top n items sorted in descending order
    top_items_by_times_ordered = df['item_name'].value_counts().nlargest(top_dishes_number).sort_values() # Times ordered
    top_items_by_quantity_ordered = df.groupby('item_name')['item_quantity'].sum().nlargest(top_dishes_number).sort_values() # Quantity ordered
    top_plant_based_items_by_times_ordered = plant_df['item_name'].value_counts().nlargest(top_dishes_number).sort_values() # Plant based, times ordered
    top_plant_based_items_by_quantity_ordered = plant_df.groupby('item_name')['item_quantity'].sum().nlargest(top_dishes_number).sort_values() # Plant based, quantity ordered

    dfs = [top_items_by_times_ordered, top_items_by_quantity_ordered, top_plant_based_items_by_times_ordered, top_plant_based_items_by_times_ordered]
    titles = ['Total by Times Ordered', 'Total by Quantity Ordered', 'Plant-Based by Times Ordered', 'Plant-Based by Quantity Ordered']
    xlabs = ['Times Ordered','Quantity Ordered','Times Ordered','Quantity Ordered']
    colors = ['blue','cyan', 'green', 'lime'] # currently not used

    to_visualize = zip(dfs, titles, xlabs, colors)

    for j, tup in enumerate(to_visualize):

        # Unpack
        df, title, xlab, color = tup

        # Overwrite to get custom colors within each graph
        colors = pd.merge(df, colors_df, left_index=True, right_index=True, how='left')['colors']

        colors = colors.fillna("#708086")

        # print(pd.concat([colors,df]))

        # Plot in the appropriate subplot as a horizontal bar chart
        axs[4*i + j].barh(df.index, df, color=colors)
        axs[4*i + j].set_title(f'{location_id}\n{title}')
        
        # Set y-axis label and make y-labels smaller
        axs[4*i + j].tick_params(axis='y', labelsize=10)
        axs[4*i + j].set_xlabel(xlab)

# Add a global title at the top of the figure
fig.suptitle('Distribution of Plant-Based and Total Item Orders in Each Restaurant', fontsize=16)

# Adjust layout and save the figure
plt.tight_layout()
plt.subplots_adjust(top=0.9)
# plt.subplots_adjust(top=0.97)  # Adjust the top margin to make room for the global title
plt.savefig('visuals/Restaurant Both Sales Distributions.png', bbox_inches='tight')